In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

In [0]:
database_name = "retail_sales_dw"
table_name = "shop_name"

target_table = f"{database_name}.{table_name}"
bronze_table_name = f"{target_table}_bronze"
silver_table_name = f"{target_table}_silver"
bad_record_table_name = f"{target_table}_bad_record"

schema_detail = {
    "shop_id": "string",
    "shop_name": "string",
    "branch_name": "string",
    "file_dt": "date"
}

data_col = list(schema_detail.keys())

keys = ['shop_id']

invalid_rule = {
    "int": r"^[0-9]+$",
    "date": r"^\d{4}-\d{2}-\d{2}$"
}

write_mode = "overwrite"

In [0]:
bronze_df = (
    spark.table(bronze_table_name)
    .select(monotonically_increasing_id().alias("_sk"), *data_col)
    )
bronze_df.display()

In [0]:
def get_reason(df:DataFrame) -> DataFrame:
    control_col = [col_name for col_name in df.columns if col_name.startswith("_") and col_name != "_sk"]
    data_col = [col_name for col_name in df.columns if not col_name.startswith("_")]
    or_statement = " OR ".join([col_name for col_name in control_col])
    return (
        df
        .filter(or_statement)
        .melt(
            ids = [*data_col,"_sk"]
            ,values = control_col
            ,variableColumnName= "reason"
            ,valueColumnName= "status"
            )
        .filter(col("status") == True)
        .groupBy(*data_col,"_sk")
        .agg(collect_list("reason").alias("reason"))
        )

In [0]:
invalid_col = {
    f"_is_{col_name}_invalid":~coalesce(col(col_name).rlike(invalid_rule[col_type]),lit(False)) 
    for col_name,col_type in schema_detail.items() if col_type not in ["string"]
    }

invalid_df = (
    bronze_df
    .withColumns(invalid_col)
    .transform(get_reason)
    )
invalid_df.display()

In [0]:
#is key null
key_null_statement = { f'_is_{col_name}_null':col(col_name).isNull() for col_name in keys}

key_null_df = (
    bronze_df.withColumns(key_null_statement)
    .transform(get_reason)
    )
key_null_df.display()

In [0]:
#is duplicate

partition_by_all = Window.partitionBy(*data_col).orderBy("_sk")
partition_by_key = Window.partitionBy(*keys)

bronze_not_null_df = bronze_df.join(key_null_df,['_sk'],"left_anti")

is_row_duplicate_df = (
    bronze_not_null_df
    .withColumn("rn",row_number().over(partition_by_all))
    .filter(col("rn") > 1)
    .drop("rn")
    .withColumn("reason",array(lit("_row_duplicate")))
    )

is_key_duplicate_df = (
    bronze_not_null_df
    .join(is_row_duplicate_df,['_sk'],"left_anti")
    .withColumn("count",count("*").over(partition_by_key))
    .filter(col("count") > 1)
    .drop("count")
    .withColumn("reason",array(lit("_key_duplicate")))
)
duplicate_df = (
    is_row_duplicate_df
    .unionByName(is_key_duplicate_df)
)

duplicate_df.display()

In [0]:
#combine bad record
bad_record_df = (
    invalid_df
    .unionByName(key_null_df)
    .unionByName(duplicate_df)
    .groupBy(*data_col,"_sk")
    .agg(flatten(collect_list("reason")).alias("reason"))
    )
bad_record_df.display()

In [0]:
add_control_col = {"_load_dt":current_date(),"_load_dttm":current_timestamp()}
cast_statement = [
    expr(
        f"try_cast(`{col_name}` AS {col_type})"
    ).alias(col_name)
    for col_name, col_type in schema_detail.items()
]

final_result_df = (
    bronze_df
    .join(bad_record_df,['_sk'],"left_anti")
    .select(cast_statement)
    .withColumns(add_control_col)
    )

final_result_df.display()

In [0]:
# final_result_df.write.mode(write_mode).saveAsTable(silver_table_name)
# bad_record_df.write.mode(write_mode).saveAsTable(bad_record_table_name)

In [0]:
(
    final_result_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table_name)
)

(
    bad_record_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bad_record_table_name)
)